# Integrated Fairness Pipeline Demo

This notebook demonstrates the complete three-step integrated workflow:

1. **Baseline Measurement** - Audit raw data for fairness issues
2. **Transform Data + Train Model** - Apply bias mitigation pipeline and train fairness-aware model
3. **Final Validation** - Compare post-training metrics to baseline and validate against threshold

The integrated workflow combines the Measurement, Pipeline, and Training modules into a unified end-to-end system.


## Setup and Imports


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

from fairness_pipeline_dev_toolkit.pipeline.config import load_config
from fairness_pipeline_dev_toolkit.integration.orchestrator import execute_workflow
from fairness_pipeline_dev_toolkit.integration.mlflow_logger import log_workflow_results
from fairness_pipeline_dev_toolkit.metrics import FairnessAnalyzer

print("Imports successful!")


## Data Preparation

Load your dataset. For this demo, we'll use a sample dataset with features, target, and sensitive attributes.


In [ ]:
# Load your dataset
# Replace this path with your actual data file
data_path = "dev_sample_train.csv"  # or your dataset path

if Path(data_path).exists():
    df = pd.read_csv(data_path)
    print(f"Loaded dataset: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    df.head()
else:
    print(f"Data file not found: {data_path}")
    print("Creating a synthetic dataset for demonstration...")
    
    # Create synthetic data
    np.random.seed(42)
    n_samples = 1000
    
    # Features
    df = pd.DataFrame({
        'f0': np.random.randn(n_samples),
        'f1': np.random.randn(n_samples),
        'f2': np.random.randn(n_samples),
        'f3': np.random.randn(n_samples),
    })
    
    # Sensitive attribute (binary)
    df['sensitive'] = np.random.choice(['A', 'B'], size=n_samples, p=[0.6, 0.4])
    
    # Target with some bias
    bias = (df['sensitive'] == 'B').astype(int) * 0.3
    df['y'] = ((df['f0'] + df['f1'] + bias + np.random.randn(n_samples) * 0.1) > 0).astype(int)
    
    print(f"Created synthetic dataset: {df.shape}")
    df.head()


## Configuration File

Create a configuration file that specifies:
- Pipeline transformers (bias mitigation steps)
- Training method and parameters
- Primary fairness metric
- Validation threshold


In [ ]:
# Create config file
config_content = """
sensitive: ["sensitive"]
alpha: 0.05
proxy_threshold: 0.30

# Pipeline: bias mitigation transformers
pipeline:
  - name: reweigh
    transformer: "InstanceReweighting"
    params: {}

# Training: fairness-aware model training
training:
  method: "reductions"  # Options: "reductions", "regularized", "lagrangian"
  target_column: "y"
  params:
    constraint: "demographic_parity"  # or "equalized_odds"
    eps: 0.01
    T: 50

# Validation
fairness_metric: "demographic_parity_difference"
validation_threshold: 0.05  # Maximum allowed unfairness
"""

# Save config
config_path = "integrated_config.yml"
with open(config_path, "w") as f:
    f.write(config_content)

print("Configuration file created:")
print(config_content)


## Execute Integrated Workflow

Run the complete three-step workflow using the orchestrator.


In [ ]:
# Load config
config = load_config(config_path)

# Execute the complete workflow
print("Executing integrated workflow...")
print("=" * 60)

result = execute_workflow(
    config=config,
    df=df,
    output_dir="artifacts/integrated_workflow",
    min_group_size=30,
    train_size=0.8,
)

print("\nWorkflow completed!")


## Results and Validation

Display the workflow results and validation status.


In [ ]:
print("\n" + "=" * 60)
print("WORKFLOW RESULTS")
print("=" * 60)

print(f"\nValidation: {result.validation_result.message}")
print(f"\nValidation Status: {'PASSED' if result.validation_result.passed else 'FAILED'}")

print(f"\nBaseline Metrics:")
for metric_name, metric_value in result.baseline_metrics.items():
    if hasattr(metric_value, "value"):
        print(f"  {metric_name}: {metric_value.value:.4f}")
    else:
        print(f"  {metric_name}: {metric_value}")

print(f"\nFinal Metrics:")
for metric_name, metric_value in result.final_metrics.items():
    if hasattr(metric_value, "value"):
        print(f"  {metric_name}: {metric_value.value:.4f}")
    else:
        print(f"  {metric_name}: {metric_value}")

print(f"\nImprovement: {result.validation_result.improvement:.4f}")
print(f"  (Negative values indicate reduction in unfairness)")


## MLflow Integration

Log the complete workflow results to MLflow for tracking and comparison.


In [ ]:
# Optional: Log to MLflow
try:
    logged = log_workflow_results(
        result,
        config_path=config_path,
        experiment_name="fairness_integrated_workflow",
        run_name="demo_run",
    )
    
    if logged:
        print("✓ Results logged to MLflow")
        print("  - Metrics: accuracy, fairness metrics, validation results")
        print("  - Artifacts: model, config.yml, workflow results JSON")
    else:
        print("MLflow not available (skipped)")
except Exception as e:
    print(f"MLflow logging failed: {e}")
    print("(This is optional - workflow completed successfully)")


## Using the CLI Command

You can also run the integrated workflow from the command line:

```bash
fairpipe run-pipeline \
    --config integrated_config.yml \
    --csv dev_sample_train.csv \
    --output-dir artifacts/integrated_workflow \
    --mlflow-experiment fairness_integrated_workflow
```

This will:
1. Run baseline measurement
2. Apply pipeline transformations and train model
3. Validate against threshold
4. Save all artifacts and optionally log to MLflow
